In [0]:
import requests
import json
from datetime import datetime

# 1. Leemos los parámetros de los Widgets
cod_indicador = dbutils.widgets.get("indicador_code")
nombre_carpeta = dbutils.widgets.get("folder_name")

class SocioDataExtractor:
    def __init__(self, folder_name):
        # Configuramos la URL base del Banco Mundial
        self.base_url = "https://api.worldbank.org/v2/country/all/indicator/"
        
        # Generamos el timestamp actual (AñoMesDía_HoraMinuto)
        # Ejemplo: 20260325_1705
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        
        # Definimos la ruta del archivo incluyendo el timestamp
        # Así evitamos el overwrite y mantenemos un histórico en Landing
        self.landing_path = f"/Volumes/socioeconomics/landing/world_bank/{folder_name}_{self.timestamp}.json"
        
        # Creamos la carpeta si no existe para evitar errores de ruta
        dbutils.fs.mkdirs(f"/Volumes/socioeconomics/landing/world_bank/")

    def obtener_datos(self, cod_indicador):
        try:
            todos_los_datos = []
            pagina = 1
            total_paginas = 1 
            rows_per_page = 1000 # Punto dulce para velocidad y estabilidad

            print(f"🚀 Iniciando extracción de: {cod_indicador}")
            print(f"📂 Destino: {self.landing_path}")

            while pagina <= total_paginas:
                url = f"{self.base_url}{cod_indicador}?format=json&per_page={rows_per_page}&page={pagina}"
                
                res = requests.get(url, timeout=30)
                res.raise_for_status()
                payload = res.json()

                # La primera página nos dice cuántas hay en total
                if pagina == 1:
                    total_paginas = payload[0]['pages']
                    registros_totales = payload[0]['total']
                    print(f"📊 Total registros detectados: {registros_totales} en {total_paginas} páginas.")

                # Agregamos los datos a nuestra lista maestra
                todos_los_datos.extend(payload[1])
                
                if pagina % 5 == 0 or pagina == total_paginas:
                    print(f"✅ Procesada página {pagina} de {total_paginas}...")
                
                pagina += 1

            # Guardamos el JSON masivo (ahora sí, el archivo es único por el timestamp)
            dbutils.fs.put(self.landing_path, json.dumps(todos_los_datos), overwrite=True)
            
            print(f"🏁 Extracción finalizada con éxito. Registros totales: {len(todos_los_datos)}")
            return self.landing_path

        except Exception as e:
            print(f"❌ Error crítico en la extracción: {e}")
            return None

In [0]:
# 1. Instanciamos la clase con el nombre de carpeta que viene del widget
extractor = SocioDataExtractor(folder_name=nombre_carpeta)

# 2. Ejecutamos el método con el código de indicador que viene del widget
resultado = extractor.obtener_datos(cod_indicador=cod_indicador)

# 3. Control de flujo para el DAG
if resultado:
    print(f"🚀 Proceso finalizado con éxito para el indicador: {cod_indicador}")
else:
    raise ValueError(f"Fallo crítico en la extracción del indicador {cod_indicador}")